In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import kagglehub
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [ ]:
# Download latest version
path = kagglehub.dataset_download("drsaeedmohsen/ucihar-dataset")
path = '/kaggle/input/ucihar-dataset/UCI-HAR Dataset'

print("Path to dataset files:", path)

subject_train_file = os.path.join(path, 'train', 'subject_train.txt')
subjects_train = np.loadtxt(subject_train_file, dtype=int)

subjects_test_file = os.path.join(path, 'test', 'subject_test.txt')
subjects_test = np.loadtxt(subjects_test_file, dtype=int)

# Load raw sensor data
body_acc_x = np.loadtxt(f'{path}/train/Inertial Signals/body_acc_x_train.txt')
body_acc_y = np.loadtxt(f'{path}/train/Inertial Signals/body_acc_y_train.txt')
body_acc_z = np.loadtxt(f'{path}/train/Inertial Signals/body_acc_z_train.txt')
body_gyro_x = np.loadtxt(f'{path}/train/Inertial Signals/body_gyro_x_train.txt')
body_gyro_y = np.loadtxt(f'{path}/train/Inertial Signals/body_gyro_y_train.txt')
body_gyro_z = np.loadtxt(f'{path}/train/Inertial Signals/body_gyro_z_train.txt')

# Stack features along the third dimension
X_train = np.stack([body_acc_x, body_acc_y, body_acc_z, body_gyro_x, body_gyro_y, body_gyro_z], axis=2)

print("Training data shape:", X_train.shape)

y_train = np.loadtxt(f'{path}/train/y_train.txt')
y_train = y_train - 1
y_train = y_train.astype(int)

print("Training labels shape:", y_train.shape)

# Test data
body_acc_x = np.loadtxt(f'{path}/test/Inertial Signals/body_acc_x_test.txt')
body_acc_y = np.loadtxt(f'{path}/test/Inertial Signals/body_acc_y_test.txt')
body_acc_z = np.loadtxt(f'{path}/test/Inertial Signals/body_acc_z_test.txt')
body_gyro_x = np.loadtxt(f'{path}/test/Inertial Signals/body_gyro_x_test.txt')
body_gyro_y = np.loadtxt(f'{path}/test/Inertial Signals/body_gyro_y_test.txt')
body_gyro_z = np.loadtxt(f'{path}/test/Inertial Signals/body_gyro_z_test.txt')

# Stack features along the third dimension
X_test = np.stack([body_acc_x, body_acc_y, body_acc_z, body_gyro_x, body_gyro_y, body_gyro_z], axis=2)

print("Testing data shape:", X_test.shape)

y_test = np.loadtxt(f'{path}/test/y_test.txt')
y_test = y_test - 1
y_test = y_test.astype(int)

print("Training labels shape:", y_test.shape)

Using Colab cache for faster access to the 'ucihar-dataset' dataset.
Path to dataset files: /kaggle/input/ucihar-dataset/UCI-HAR Dataset
Training data shape: (7352, 128, 6)
Training labels shape: (7352,)
Testing data shape: (2947, 128, 6)
Training labels shape: (2947,)


In [ ]:
assert len(subjects_train) == len(X_train)

unique_subjects = np.unique(subjects_train)

train_subjects, val_subjects = train_test_split(
    unique_subjects,
    test_size=0.2,
    random_state=45
)


train_mask = np.isin(subjects_train, train_subjects)
val_mask  = np.isin(subjects_train, val_subjects)



X_val  = X_train[val_mask]
X_train = X_train[train_mask]


y_val = y_train[val_mask]
y_train = y_train[train_mask]

subjects_train_split = subjects_train[train_mask]
subjects_val_split  = subjects_train[val_mask]


overlap = np.intersect1d(subjects_train_split, subjects_val_split)
print("Subject overlap:", len(overlap))

Subject overlap: 0


In [ ]:
train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

def flatten_windows(X):
    return X.reshape(X.shape[0], -1)

X_train_f = flatten_windows(X_train)
X_val_f   = flatten_windows(X_val)
X_test_f  = flatten_windows(X_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train_f, dtype=torch.float32)
Y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val_f, dtype=torch.float32)
Y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_t = torch.tensor(X_test_f, dtype=torch.float32)
Y_test_t = torch.tensor(y_test, dtype=torch.long)

batch_size = 640

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=batch_size)
test_loader  = DataLoader(TensorDataset(X_test_t, Y_test_t), batch_size=batch_size)


torch.manual_seed(42)

## Part 11

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.001
num_epochs = 60
weight_decay = 1e-4
batch_size = 64

In [ ]:
class MyGRU(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
    super().__init__()
    self.num_layers = num_layers
    self.hidden_size = hidden_size

    self.gru = nn.GRU(
        input_size,
        hidden_size,
        num_layers,
        batch_first=True,
        dropout=dropout
    )

    # input shape: (batch_size, window, features)
    self.fc = nn.Linear(hidden_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, output_size)

  def forward(self, x):

    # initialize hidden state
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

    out, _ = self.gru(x, h0)

    # out shape: (batch_size, window, hidden_size)
    out = out[:, -1, :]   # take last timestep

    out = self.fc(out)
    out = self.fc2(out)

    return out

In [ ]:
'''model = MyLSTM().to(device)
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()'''

'model = MyLSTM().to(device)\noptimizer = optim.Adam(model.parameters())\ncriterion = nn.CrossEntropyLoss()'

## Part 12


In [ ]:
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))

val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                            torch.tensor(y_val, dtype=torch.long))

test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

input_size = 6
hidden_size = 64
num_layers = 2
output_size = 6
dropout = 0

model = MyGRU(input_size, hidden_size, num_layers, output_size, dropout).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=lr)

scheduler = optim.lr_scheduler.ExponentialLR(
    optimizer,
    gamma=0.95
)

best_val_acc = 0

for epoch in range(num_epochs):

    model.train()
    train_loss = 0
    train_correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    train_acc = train_correct / total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.size(0)

    val_acc = val_correct / val_total

    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f}")
    print(f"Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_lstm.pt")
        print("Saved best model")

Epoch 1/60
Train Loss: 157.046 | Train Acc: 0.195
Val Loss: 49.452 | Val Acc: 0.205
Saved best model
Epoch 2/60
Train Loss: 148.831 | Train Acc: 0.237
Val Loss: 45.712 | Val Acc: 0.246
Saved best model
Epoch 3/60
Train Loss: 123.692 | Train Acc: 0.298
Val Loss: 34.293 | Val Acc: 0.344
Saved best model
Epoch 4/60
Train Loss: 107.806 | Train Acc: 0.384
Val Loss: 32.432 | Val Acc: 0.396
Saved best model
Epoch 5/60
Train Loss: 98.785 | Train Acc: 0.418
Val Loss: 30.129 | Val Acc: 0.428
Saved best model
Epoch 6/60
Train Loss: 92.603 | Train Acc: 0.453
Val Loss: 27.648 | Val Acc: 0.471
Saved best model
Epoch 7/60
Train Loss: 86.461 | Train Acc: 0.490
Val Loss: 25.356 | Val Acc: 0.531
Saved best model
Epoch 8/60
Train Loss: 82.359 | Train Acc: 0.510
Val Loss: 24.900 | Val Acc: 0.516
Epoch 9/60
Train Loss: 78.436 | Train Acc: 0.520
Val Loss: 23.998 | Val Acc: 0.548
Saved best model
Epoch 10/60
Train Loss: 75.300 | Train Acc: 0.556
Val Loss: 22.938 | Val Acc: 0.548
Epoch 11/60
Train Loss: 71.46

## Part 13

In [ ]:
model.load_state_dict(torch.load("best_lstm.pt"))
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

cm = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

Confusion Matrix:
[[433  39  24   0   0   0]
 [ 18 439   9   0   2   3]
 [ 15  35 370   0   0   0]
 [  0   0   0  29 157 305]
 [  2   1   0  20 287 222]
 [  0   0   0   6 103 428]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.87      0.90       496
           1       0.85      0.93      0.89       471
           2       0.92      0.88      0.90       420
           3       0.53      0.06      0.11       491
           4       0.52      0.54      0.53       532
           5       0.45      0.80      0.57       537

    accuracy                           0.67      2947
   macro avg       0.70      0.68      0.65      2947
weighted avg       0.69      0.67      0.64      2947



### Error Analysis
The model achieves excellent performance on the dynamic activities (classes 0–2: walking, walking upstairs, and walking downstairs), with F1-scores of 0.89–0.90 and very little cross-confusion between them. In contrast, the static postures (classes 3–5: sitting, standing, and laying) show substantial errors: sitting is almost always misclassified as laying (305 cases) or standing (157 cases), standing is frequently confused with laying and sitting, and laying has high recall but low precision due to being over-predicted for the other two static classes.    

These confusions are highly plausible because the body-acceleration and gyroscope signals for stationary activities are all dominated by near-zero movement with only subtle differences in gravitational orientation and minor postural sway, which a GRU relying solely on temporal sequences struggles to disambiguate without additional spatial or visual context. Overall, the 67% test accuracy is driven almost entirely by the strong separation of dynamic vs. static behaviors, while the static group remains the clear limitation of the current architecture.

## Part 14

In [ ]:
import time
import torch.nn.functional as F

# Move model to CPU for the latency test
cpu_device = torch.device("cpu")

model.to(cpu_device)
model.eval()

def predict_activity(window):
    if not isinstance(window, torch.Tensor):
        window_t = torch.tensor(window, dtype=torch.float32)
    else:
        window_t = window.clone().detach().float()

    # Add batch dimension
    if window_t.dim() == 2:
        window_t = window_t.unsqueeze(0)
    window_t = window_t.to(cpu_device)

    # Run inference
    with torch.no_grad():
        logits = model(window_t)
        # Convert logits to probabilities
        probs = F.softmax(logits, dim=1)
        label = torch.argmax(probs, dim=1).item()
    return label, probs.numpy()[0]

# Latency measurement
num_samples = 500
# Grab the first 500 windows from the test set
sample_windows = X_test[:num_samples]

latencies = []

for i in range(num_samples):
    window = sample_windows[i]

    # Start timer
    start_time = time.perf_counter()

    label, probs = predict_activity(window)

    # End timer
    end_time = time.perf_counter()

    # Calculate latency in seconds
    latencies.append(end_time - start_time)

# Calculate and display average latency
average_latency_sec = sum(latencies) / num_samples
average_latency_ms = average_latency_sec * 1000

print(f"Tested over {num_samples} windows on CPU.")
print(f"Average inference latency per window: {average_latency_ms:.2f} ms")

Tested over 500 windows on CPU.
Average inference latency per window: 6.52 ms


In terms of inference speed the model is sufficiently fast, it only takes 6.52ms to infer a windows size of 500, however if we infer without overlapping then a window of 500 would takes 10s to collect so we would only be able to infer every 10s. Window size of 100 would bring it down to 2s. One timestep is 20ms so because inference only takes 6.52ms we can consider it realtime.

## Part 15



### Model Card

**Model Name**: MyGRU (2-layer GRU for UCI-HAR)  
**Input**: Raw 128 × 6 inertial window (3 body acceleration + 3 body gyroscope axes)  
**Output**: 6-class activity logits (Walking, Walking Upstairs, Walking Downstairs, Sitting, Standing, Laying)  
**Architecture**: 2-layer GRU (hidden size 64), batch-first, followed by two linear layers, trained with gradient clipping and exponential LR decay (γ=0.95).  
**Training**: 80/20 subject-split of UCI-HAR training set (subject-independent), Adam (lr=0.001), CrossEntropyLoss, weight decay 1e-4, dropout=0.

### Intended Use
This model is intended for **real-time human activity recognition** on smartphone or wearable inertial sensors (accelerometer + gyroscope) to detect the six UCI-HAR activities. It is suitable for health/fitness apps, elderly monitoring, or context-aware mobile features where the user is carrying the device in a pocket or on the waist.

### Non-Intended Use
- Not for clinical/medical diagnosis (e.g., fall detection or gait analysis for pathology).  
- Not suitable for activities outside the six classes (running, cycling, driving, etc.).  
- Do not deploy on devices without 50 Hz sampling or without the exact sensor placement used in UCI-HAR (waist-mounted).  
- Not intended for multi-person or outdoor environments with heavy vibration.

### Evaluation Protocol
- **Subject-independent split**: Training/validation subjects completely disjoint from test subjects (30 subjects total, 24 train/val, 6 test).  
- Best model selected by highest validation accuracy (0.713 at epoch 59).  
- Final evaluation on the official UCI-HAR test set (2,947 windows).  
- Metrics: accuracy, per-class precision/recall/F1, confusion matrix.

### Known Failure Modes
- **Static posture confusion**: Sitting, Standing, and Laying are frequently mixed (especially Sitting → Laying and Standing → Laying) because these activities produce near-identical near-zero acceleration signals.  
- **Low recall for Sitting** (only 6%): the model rarely predicts Sitting correctly.  
- Performance drops if sensor orientation changes or if the device is held in hand instead of pocket/waist.  
- No robustness to unseen subjects with very different gait patterns or body mass.

### Privacy / Ethics Note
This model only processes anonymous inertial sensor data (no camera, microphone, or location). However, activity patterns can still indirectly reveal sensitive information (sleep habits, mobility limitations, or daily routines). Users should be informed and give explicit consent before data collection. The model does not store raw sensor streams; only the predicted label is returned.

